> (중요) 여러분 구글 드라이브에 최소 7GB 이상은 확보되어 있어야 합니다!

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#### `normalize_path`는?

`normalize_path`는 파일 경로에 포함된 **한글(유니코드) 처리 방식**을 통일하여, 경로를 찾지 못하는 에러를 방지하기 위한 함수입니다.

특히 **Google Colab**이나 **Mac, Windows** 사이에서 데이터를 주고받을 때 한글 폴더명이 깨져서 발생하는 `File Not Found` 에러를 잡는 데 필수적입니다.

<br>

##### 왜 사용하나요? (NFC vs NFD)

한글을 컴퓨터가 인식하는 방식은 크게 두 가지입니다.

* **NFC (Windows 스타일):** '강'을 '강'이라는 하나의 글자로 저장합니다.
* **NFD (Mac/Unix 스타일):** '강'을 'ㄱ', 'ㅏ', 'ㅇ'으로 쪼개서 저장합니다.

사람 눈에는 똑같이 "초급 프로젝트"라고 보이지만, 컴퓨터 입장에서는 글자 조합 방식이 다르면 **완전히 다른 경로**로 인식합니다. `normalize_path` 는 이를 **NFC(표준 방식)** 로 강제 통일해주는 역할을 합니다.

In [2]:
############################################################
# 0. 라이브러리 임포트 & 경로 설정
############################################################
import os
import json
import random
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import unicodedata
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torchvision
from torchvision.transforms import v2
from torchvision import tv_tensors
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import wandb
from tqdm import tqdm

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

# 경로는 환경에 맞게 수정
# train_images, test_images
extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/") 

TRAIN_JSON_PATH = os.path.join(extract_path, "merged_annotations_train_final.json")
TEST_JSON_PATH = os.path.join(extract_path, "merged_annotations_test_final.json")
TRAIN_IMG_DIR = os.path.join(extract_path, "train_images")
TEST_IMG_DIR  = os.path.join(extract_path, "test_images")

# merged_annotation json 경로
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

############################################################
# 1. 병합된 JSON 파일을 읽어서 DataFrame으로 만들기
############################################################

def build_df_from_merged_json(json_path, img_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 1) 이미지 정보 매핑 (id -> file_name)
    id_to_fname = {img["id"]: img["file_name"] for img in data["images"]}

    records = []
    # 2) 어노테이션 순회
    for ann in data["annotations"]:
        img_id_coco = ann["image_id"]
        file_name = id_to_fname.get(img_id_coco)
        
        if file_name is None: continue
        
        img_path = os.path.join(img_dir, file_name)
        
        # 실제 이미지 파일이 있는지 확인 (선택 사항이지만 안전함)
        if not os.path.exists(img_path):
            continue

        x, y, w, h = ann["bbox"]
        if w <= 1 or h <= 1: continue # 너무 작은 박스는 무시
        
        records.append({
            "image_path": img_path,
            "image_id": os.path.splitext(file_name)[0], # 파일명을 ID로 사용
            "category_id": int(ann["category_id"]),
            "bbox_x": float(x),
            "bbox_y": float(y),
            "bbox_w": float(w),
            "bbox_h": float(h),
        })

    return pd.DataFrame(records)

# 실행
df = build_df_from_merged_json(TRAIN_JSON_PATH, TRAIN_IMG_DIR)
print(f"✅ 학습 데이터 로드 완료: {len(df)} 개의 객체 탐지됨")

✅ 학습 데이터 로드 완료: 4526 개의 객체 탐지됨


In [3]:
############################################################
# 2. category_id 매핑 (겉으로는 안 바꾸고, 모델 내부에서만 사용)
############################################################

# 원본 category_id 집합
unique_cats = sorted(df["category_id"].unique())
print("고유 category_id 개수:", len(unique_cats))

# ◀ 보완: 추후 WandB에서 클래스별 mAP를 확인하기 위해 이름을 리스트로 저장
# 0번은 항상 "background"여야 합니다.
class_names = ["background"] + [f"drug_{cid}" for cid in unique_cats]

# 내부용: 모델에 넣을 label (1 ~ num_classes-1), 0은 background
orig2model = {cid: i + 1 for i, cid in enumerate(unique_cats)}   # 원본 → 모델용
model2orig = {v: k for k, v in orig2model.items()}               # 모델용 → 원본

num_classes = len(unique_cats) + 1  # background 포함
print("num_classes (background 포함):", num_classes)

# ◀ 추가: 매핑이 잘 되었는지 확인 (첫 3개만)
print("Mapping Sample (Orig -> Model):", list(orig2model.items())[:3])

고유 category_id 개수: 73
num_classes (background 포함): 74
Mapping Sample (Orig -> Model): [(np.int64(1899), 1), (np.int64(2482), 2), (np.int64(3350), 3)]


In [4]:
# wandb 설치 및 로그인
!pip install wandb
import wandb
import os

wandb.login(key=())

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: codeit-project-team3 (codeit-project-team3-pilltering) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
############################################################
# wandb 실험 초기화
# ※ run_name을 바꿔가며 실험을 구분하세요 (예: "exp01-baseline", "exp02-augment")
############################################################

config = {
    "run_name"       : "exp02-mosaic-SGD",   # 실험 이름 변경 (기록용)
    "model"          : "fasterrcnn_resnet50_fpn",
    "num_epochs"     : 5,
    "batch_size"     : 4, 
    "optimizer"      : "SGD",           # ◀ "" 대신 "SGD" 입력
    "lr"             : 0.005,           # ◀ SGD 사용 시 0.005가 황금 밸런스입니다
    "momentum"       : 0.9,             # ◀ SGD의 단짝 친구 (안정성 강화)
    "weight_decay"   : 0.0005, 
    "lr_step_size"   : 5,
    "lr_gamma"       : 0.1,
    "score_threshold": 0.01,
    "mosaic_ratio"   : 0.5,            
    "img_size"       : 640,                
    "num_classes"    : num_classes,
}

wandb.init(
    entity="codeit-project-team3-pilltering",
    project="Pill-터링Project_초급프로젝트",
    name=config["run_name"],
    config=config,
)

# ◀ 추가: config 값을 실제 변수로 할당하여 코드 전체에서 사용하기 편하게 함
NUM_EPOCHS = config["num_epochs"]
BATCH_SIZE = config["batch_size"]
LEARNING_RATE = config["lr"]

print(f"✅ wandb 초기화 완료 | run: {wandb.run.name}")
# 모델 저장 시 사용할 이름 미리 정의
SAVE_NAME = f"{config['run_name']}_best.pth"
print(f"🚀 모델은 {SAVE_NAME}으로 저장될 예정입니다.")

✅ wandb 초기화 완료 | run: exp02-mosaic-SGD
🚀 모델은 exp02-mosaic-SGD_best.pth으로 저장될 예정입니다.


In [6]:
############################################################
# 3. 데이터 정제: 이상 Bbox 사전 제거 (수정: 모자이크/Resize 대비)
############################################################

# 다시 로드 (최신 상태 유지)
df = build_df_from_merged_json(TRAIN_JSON_PATH, TRAIN_IMG_DIR)

# ✅ 모델 학습 규격 설정 (640으로 학습할 예정)
IMG_W, IMG_H = 1280, 1280  # 원본 크기

before = len(df)

# 1) 너무 작거나 이미지 밖으로 나가는 Bbox 제거
# ◀ 수정: 박스 크기가 2~5픽셀 미만인 '너무 작은' 객체는 모델이 노이즈로 인식하므로 제거 임계치를 살짝 높임 (1 -> 2)
df = df[
    (df["bbox_w"] > 2) & 
    (df["bbox_h"] > 2) &
    (df["bbox_x"] >= 0) &
    (df["bbox_y"] >= 0) &
    (df["bbox_x"] + df["bbox_w"] <= IMG_W) &
    (df["bbox_y"] + df["bbox_h"] <= IMG_H)
].reset_index(drop=True)

# 2) ◀ 추가: 한 이미지에 객체가 아예 없는 데이터가 생길 수 있으므로 체크
# (모자이크 증강 시 빈 이미지가 섞이면 mAP가 급격히 떨어집니다.)
valid_image_ids = df["image_id"].unique()
df = df[df["image_id"].isin(valid_image_ids)].reset_index(drop=True)

print(f"✅ 정제 완료: 제거된 bbox: {before - len(df)}개 | 남은 bbox: {len(df)}개")
print(f"✅ 유효 이미지 수: {len(valid_image_ids)}장")


✅ 정제 완료: 제거된 bbox: 2개 | 남은 bbox: 4524개
✅ 유효 이미지 수: 1489장


In [7]:
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 1. Train Transforms: Mosaic 이후에 적용될 추가 증강
# ◀ 수정 포인트: Mosaic과 중복되는 강한 변형은 줄이고, 색감 위주로 보정하여 mAP 향상
train_transforms = A.Compose([
    # 640 리사이즈는 Dataset 클래스에서 Mosaic 처리 시 함께 수행하므로 여기선 보조적 역할
    A.Resize(640, 640),
    
    # ◀ 수정: Mosaic 자체가 위치 변형이므로, 추가 변형은 적절한 p값(0.5)으로 조정
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.1, rotate_limit=15, border_mode=0, p=0.5),
    
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3), # 약물 데이터는 상하반전보다 좌우반전이 더 빈번함
    
    # ◀ 핵심: 약물 데이터의 특징인 색상/질감을 살리는 증강 추가
    A.OneOf([
        A.CLAHE(clip_limit=2.0, p=0.5),
        A.Sharpen(p=0.5),
        A.Emboss(p=0.5),
    ], p=0.3),
    
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
    A.GaussNoise(p=0.2),
    
    # 모델 학습 규격에 맞춘 정규화
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], bbox_params=A.BboxParams(
    format='coco', # Dataset에서 주는 형식을 coco로 유지
    label_fields=['category_ids'],
    min_area=2.0,       # ◀ 수정: 너무 작은 박스는 학습에서 제외하여 정밀도 향상
    min_visibility=0.2,  # ◀ 수정: 박스의 20% 이상이 보일 때만 학습 (Mosaic 절단 대비)
    clip=True
))

# 2. Val Transforms: 평가는 정직하게!
val_transforms = A.Compose([
    A.Resize(640, 640),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], bbox_params=A.BboxParams(
    format='coco', 
    label_fields=['category_ids'], 
    clip=True
))

print("✨ mAP 0.9 전용 Transforms 설정 완료")



✨ mAP 0.9 전용 Transforms 설정 완료


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [8]:
import random

class OralDrugDataset(Dataset):
    def __init__(self, df, orig2model, transforms=None, mosaic_ratio=0.5, img_size=640):
        self.df = df.reset_index(drop=True)
        self.orig2model = orig2model
        self.transforms = transforms
        self.mosaic_ratio = mosaic_ratio
        self.img_size = img_size
        self.grouped = {img_id: grp for img_id, grp in self.df.groupby("image_id")}
        self.image_ids = list(self.grouped.keys())

    def __len__(self):
        return len(self.image_ids)

    # ◀ 추가: 개별 이미지 로드 함수 (모자이크에서 재사용)
    def load_image_and_anno(self, idx):
        image_id = self.image_ids[idx]
        df_img = self.grouped[image_id]
        img_path = df_img["image_path"].iloc[0]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        bboxes, category_ids = [], []
        for _, row in df_img.iterrows():
            bboxes.append([row["bbox_x"], row["bbox_y"], row["bbox_w"], row["bbox_h"]])
            category_ids.append(self.orig2model[int(row["category_id"])])
        return image, bboxes, category_ids

    # ◀ 핵심: 모자이크 생성 로직 (4분할 합체)
    def load_mosaic(self, idx):
        s = self.img_size
        # 4장의 이미지 선택 (현재 인덱스 + 랜덤 3장)
        indices = [idx] + [random.randint(0, len(self.image_ids) - 1) for _ in range(3)]
        
        # 2x2 모자이크 캔버스 (배경색 114)
        mosaic_img = np.full((s, s, 3), 114, dtype=np.uint8)
        # 모자이크 분할 기준점 (중심)
        xc, yc = [int(random.uniform(s * 0.3, s * 0.7)) for _ in range(2)]
        
        merged_bboxes, merged_labels = [], []

        for i, index in enumerate(indices):
            img, bboxes, labels = self.load_image_and_anno(index)
            h, w = img.shape[:2]

            # 각 사분면 배치 영역 설정 (i=0:좌상, 1:우상, 2:좌하, 3:우하)
            if i == 0:  # 좌상
                x1a, y1a, x2a, y2a = 0, 0, xc, yc
                x1b, y1b, x2b, y2b = w - xc, h - yc, w, h
            elif i == 1:  # 우상
                x1a, y1a, x2a, y2a = xc, 0, s, yc
                x1b, y1b, x2b, y2b = 0, h - yc, s - xc, h
            elif i == 2:  # 좌하
                x1a, y1a, x2a, y2a = 0, yc, xc, s
                x1b, y1b, x2b, y2b = w - xc, 0, w, s - yc
            elif i == 3:  # 우하
                x1a, y1a, x2a, y2a = xc, yc, s, s
                x1b, y1b, x2b, y2b = 0, 0, s - xc, s - yc

            # 이미지 크기 및 오프셋 계산 (Resize 및 Crop 처리)
            # (계산 편의상 원본을 타겟 영역 크기에 맞게 resize하여 배치)
            img_part = cv2.resize(img, (x2a - x1a, y2a - y1a))
            mosaic_img[y1a:y2a, x1a:x2a] = img_part
            
            # Bbox 좌표 변환 (원본 1280 -> 640 스케일 조정 후 위치 오프셋 추가)
            pad_x, pad_y = x1a, y1a
            scale_x, scale_y = (x2a - x1a) / w, (y2a - y1a) / h
            
            for box, label in zip(bboxes, labels):
                new_x = box[0] * scale_x + pad_x
                new_y = box[1] * scale_y + pad_y
                new_w = box[2] * scale_x
                new_h = box[3] * scale_y
                merged_bboxes.append([new_x, new_y, new_w, new_h])
                merged_labels.append(label)

        return mosaic_img, merged_bboxes, merged_labels

    def __getitem__(self, idx):
        # 1. Mosaic 적용 여부 (Train 시에만 작동하도록 설정 가능)
        if random.random() < self.mosaic_ratio:
            image, bboxes, category_ids = self.load_mosaic(idx)
        else:
            image, bboxes, category_ids = self.load_image_and_anno(idx)

        # 2. Albumentations 적용
        if self.transforms is not None:
            transformed = self.transforms(image=image, bboxes=bboxes, category_ids=category_ids)
            image = transformed["image"]
            bboxes = transformed["bboxes"]
            category_ids = transformed["category_ids"]
        else:
            # 기본 Tensor 변환 (Resize 640 포함)
            image = cv2.resize(image, (self.img_size, self.img_size))
            image = torch.as_tensor(image / 255.0, dtype=torch.float32).permute(2, 0, 1)

        # 3. XYXY 변환 (Faster R-CNN용)
        boxes_xyxy, valid_labels = [], []
        for (x, y, w, h), label in zip(bboxes, category_ids):
            if w > 0.5 and h > 0.5: # 유효한 크기만 남김
                boxes_xyxy.append([x, y, x + w, y + h])
                valid_labels.append(label)

        boxes_tensor = (torch.tensor(boxes_xyxy, dtype=torch.float32)
                        if boxes_xyxy else torch.zeros((0, 4), dtype=torch.float32))

        target = {
            "boxes": boxes_tensor,
            "labels": torch.tensor(valid_labels, dtype=torch.int64),
            "image_id": torch.tensor([idx])
        }
        return image, target


In [9]:
############################################################
# 4. Dataset 생성 및 DataLoader 구성 (최종 통합 버전)
############################################################

# 1) 전체 데이터셋 생성 (Mosaic 비율 0.5 적용)
# 위에서 정의한 최신 MosaicOralDrugDataset 클래스를 사용합니다.
full_dataset = OralDrugDataset(
    df=df, 
    orig2model=orig2model, 
    transforms=train_transforms, 
    mosaic_ratio=config["mosaic_ratio"], # 0.5
    img_size=config["img_size"]          # 640
)

# 2) train/val split (90% train, 10% val)
# 이 부분은 주석에 있던 로직을 그대로 살리되 성능을 위해 seed를 고정합니다.
torch.manual_seed(42) 
indices = torch.randperm(len(full_dataset)).tolist()
split = int(0.9 * len(indices))
train_indices = indices[:split]
val_indices   = indices[split:]

train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
val_dataset   = torch.utils.data.Subset(full_dataset, val_indices)

# ◀ 중요: Validation은 Mosaic 없이 깨끗하게 평가해야 하므로 transforms를 교체합니다.
# Subset 내부의 dataset 객체에 접근하여 val_transforms를 적용
import copy
val_dataset.dataset = copy.deepcopy(full_dataset)
val_dataset.dataset.transforms = val_transforms
val_dataset.dataset.mosaic_ratio = 0.0 # 평가는 모자이크 없이!

# 3) collate_fn (이건 꼭 살려야 함!)
# Faster R-CNN은 이미지마다 정답 박스 개수가 달라서 이게 필수입니다.
def collate_fn(batch):
    return tuple(zip(*batch))

# 4) DataLoader 구성 (config 값 반영)
train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"], 
    shuffle=True,
    num_workers=2, # 데이터 로딩 속도 향상
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn,
)

print(f"✅ 데이터 준비 완료! Train: {len(train_loader)} steps | Val: {len(val_loader)} steps")


✅ 데이터 준비 완료! Train: 335 steps | Val: 38 steps


In [9]:
# ############################################################
# # 3. Dataset 정의
# ############################################################

# class OralDrugDataset(Dataset):
#     """
#     Faster R-CNN용 Dataset
#     - __getitem__은 (image, target) 반환
#     - image: [C,H,W] float tensor
#     - target: dict(boxes, labels, image_id, ...)
#     """
#     def __init__(self, df, orig2model, transforms=None):
#         self.df = df.reset_index(drop=True)
#         self.orig2model = orig2model
#         self.transforms = transforms

#         # 이미지 단위로 그룹을 만들기 위해 고유 image_id 리스트를 유지
#         self.image_ids = self.df["image_id"].unique().tolist()

#     def __len__(self):
#         return len(self.image_ids)

#     def __getitem__(self, idx):
#         # 1) image_id 선택
#         image_id = self.image_ids[idx]

#         # 2) 해당 image_id에 해당하는 모든 row (여러 객체) 가져오기
#         df_img = self.df[self.df["image_id"] == image_id]

#         # 3) 이미지 로드
#         img_path = df_img["image_path"].iloc[0]
#         image = Image.open(img_path).convert("RGB")

#         # 4) bbox / labels 구성
#         boxes = []
#         labels = []

#         for _, row in df_img.iterrows():
#             x = row["bbox_x"]
#             y = row["bbox_y"]
#             w = row["bbox_w"]
#             h = row["bbox_h"]

#             # [x1, y1, x2, y2] 형식으로 변환
#             x1 = x
#             y1 = y
#             x2 = x + w
#             y2 = y + h
#             boxes.append([x1, y1, x2, y2])

#             # 원본 category_id → 모델용 label로 변환
#             orig_cat = int(row["category_id"])
#             model_cat = self.orig2model[orig_cat]
#             labels.append(model_cat)

#         boxes = torch.tensor(boxes, dtype=torch.float32)
#         labels = torch.tensor(labels, dtype=torch.int64)

#         target = {
#             "boxes": boxes,
#             "labels": labels,
#             # image_id는 loss에는 크게 영향 없음, 그냥 인덱스 정도로 넣어도 무방
#             "image_id": torch.tensor([idx]),
#         }

#         if self.transforms is not None:
#             image = self.transforms(image)

#         return image, target


# ############################################################
# # 4. Transform, Dataset, DataLoader 구성
# ############################################################

# train_transforms = T.Compose([
#     # 필요하다면 여기서 RandomHorizontalFlip 등 augmentation 추가
#     T.ToTensor(),  # PIL 이미지를 [0,1] float tensor로 변환
# ])

# dataset = OralDrugDataset(df, orig2model=orig2model, transforms=train_transforms)

# # 간단히 train/val split (예: 90% train, 10% val)
# indices = torch.randperm(len(dataset)).tolist()
# split = int(0.9 * len(indices))
# train_indices = indices[:split]
# val_indices   = indices[split:]

# train_dataset = torch.utils.data.Subset(dataset, train_indices)
# val_dataset   = torch.utils.data.Subset(dataset, val_indices)

# def collate_fn(batch):
#     # detection 모델용 collate_fn: 리스트의 튜플을 튜플의 리스트로
#     return tuple(zip(*batch))

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=2,              # GPU 메모리에 맞게 조정
#     shuffle=True,
#     collate_fn=collate_fn,
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=2,
#     shuffle=False,
#     collate_fn=collate_fn,
# )

# print("train steps:", len(train_loader), "val steps:", len(val_loader))



train steps: 670 val steps: 75


In [10]:
############################################################
# 5. 모델 정의 (수정: Config 값 연동 및 Optimizer 최적화)
############################################################

# 사전학습된 Faster R-CNN 모델 로드
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

# 분류 head 교체 (74개 클래스 + 배경 1개 = 75)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, config["num_classes"])

model.to(DEVICE)



Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 240MB/s] 


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [30]:

# ◀ 수정: config의 lr과 weight_decay 사용
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=config["lr"], momentum=config["momentum"], weight_decay=config["weight_decay"])

# ◀ 수정: step_size를 config에 맞춰 유연하게 변경
lr_scheduler = optim.lr_scheduler.StepLR(
    optimizer, 
    step_size=config["lr_step_size"], 
    gamma=config["lr_gamma"]
)

############################################################
# 6. 학습 루프 (수정: WandB 실시간 로깅 추가)
############################################################

from tqdm import tqdm

for epoch in range(config["num_epochs"]):
    ##########################
    # 1) Train
    ##########################
    model.train()
    train_loss_sum = 0.0
    
    # tqdm으로 진행상황 보기
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['num_epochs']} [Train]")
    
    for images, targets in pbar:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        train_loss_sum += losses.item()
        pbar.set_postfix(loss=losses.item())

    avg_train_loss = train_loss_sum / len(train_loader)

    ##########################
    # 2) Validation (loss 측정)
    ##########################
    model.train() # Loss 계산을 위해 train 모드 유지
    val_loss_sum = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss_sum += losses.item()

    avg_val_loss = val_loss_sum / len(val_loader)

    # ◀ 핵심: WandB에 매 에포크 결과 기록
    wandb.log({
        "epoch": epoch + 1,
        "train/loss": avg_train_loss,
        "val/loss": avg_val_loss,
        "lr": optimizer.param_groups[0]['lr']
    })

    print(f"\n[Epoch {epoch+1}] train_loss: {avg_train_loss:.4f} | val_loss: {avg_val_loss:.4f} | lr: {optimizer.param_groups[0]['lr']}")

    # 스케줄러 업데이트
    lr_scheduler.step()

# ◀ 수정: 아까 정의한 SAVE_NAME으로 저장
final_save_path = os.path.join(extract_path, f"{config['run_name']}.pth")
torch.save(model.state_dict(), final_save_path)
print(f"✅ 구글 드라이브에 안전하게 저장 완료: {final_save_path}")

Epoch 1/15 [Train]: 100%|██████████| 335/335 [06:59<00:00,  1.25s/it, loss=0.657]



[Epoch 1] train_loss: 1.0823 | val_loss: 0.4091 | lr: 0.005


Epoch 2/15 [Train]: 100%|██████████| 335/335 [04:45<00:00,  1.17it/s, loss=0.59] 



[Epoch 2] train_loss: 0.5980 | val_loss: 0.2825 | lr: 0.005


Epoch 3/15 [Train]: 100%|██████████| 335/335 [04:45<00:00,  1.17it/s, loss=0.321]


...


[Epoch 13] train_loss: 0.2086 | val_loss: 0.0900 | lr: 5e-05


Epoch 14/15 [Train]: 100%|██████████| 335/335 [04:45<00:00,  1.17it/s, loss=0.281] 



[Epoch 14] train_loss: 0.2044 | val_loss: 0.0884 | lr: 5e-05


Epoch 15/15 [Train]: 100%|██████████| 335/335 [04:45<00:00,  1.17it/s, loss=0.323] 



[Epoch 15] train_loss: 0.2008 | val_loss: 0.0894 | lr: 5e-05
✅ 구글 드라이브에 안전하게 저장 완료: /content/drive/MyDrive/data/초급_프로젝트/dataset/exp04-mosaic-SGD-v1.pth


[Epoch 3] train_loss: 0.4346 | val_loss: 0.1914 | lr: 0.005
Epoch 4/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.358]

[Epoch 4] train_loss: 0.3610 | val_loss: 0.1932 | lr: 0.005
Epoch 5/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.399]

[Epoch 5] train_loss: 0.3288 | val_loss: 0.1622 | lr: 0.005
Epoch 6/20 [Train]: 100%|██████████| 335/335 [04:44<00:00,  1.18it/s, loss=0.351] 

[Epoch 6] train_loss: 0.2385 | val_loss: 0.0987 | lr: 0.0005
Epoch 7/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.195] 

[Epoch 7] train_loss: 0.2187 | val_loss: 0.0923 | lr: 0.0005
Epoch 8/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.231] 

[Epoch 8] train_loss: 0.2125 | val_loss: 0.0904 | lr: 0.0005
Epoch 9/20 [Train]: 100%|██████████| 335/335 [04:44<00:00,  1.18it/s, loss=0.312] 

[Epoch 9] train_loss: 0.2129 | val_loss: 0.0886 | lr: 0.0005
Epoch 10/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.293] 

[Epoch 10] train_loss: 0.2062 | val_loss: 0.0899 | lr: 0.0005
Epoch 11/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.237] 

[Epoch 11] train_loss: 0.2007 | val_loss: 0.0838 | lr: 5e-05
Epoch 12/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.389] 

[Epoch 12] train_loss: 0.1970 | val_loss: 0.0819 | lr: 5e-05
Epoch 13/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.167] 

[Epoch 13] train_loss: 0.1992 | val_loss: 0.0831 | lr: 5e-05
Epoch 14/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.283] 

[Epoch 14] train_loss: 0.1946 | val_loss: 0.0832 | lr: 5e-05
Epoch 15/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.343] 

[Epoch 15] train_loss: 0.1923 | val_loss: 0.0823 | lr: 5e-05
Epoch 16/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.165] 

[Epoch 16] train_loss: 0.1912 | val_loss: 0.0828 | lr: 5e-06
Epoch 17/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.257] 

[Epoch 17] train_loss: 0.1971 | val_loss: 0.0822 | lr: 5e-06
Epoch 18/20 [Train]: 100%|██████████| 335/335 [04:44<00:00,  1.18it/s, loss=0.267] 

[Epoch 18] train_loss: 0.1952 | val_loss: 0.0828 | lr: 5e-06
Epoch 19/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.162] 

[Epoch 19] train_loss: 0.1946 | val_loss: 0.0828 | lr: 5e-06
Epoch 20/20 [Train]: 100%|██████████| 335/335 [04:43<00:00,  1.18it/s, loss=0.21]  

[Epoch 20] train_loss: 0.1966 | val_loss: 0.0826 | lr: 5e-06
✅ 모델 저장 완료: exp04-mosaic-SGD-v1.pth

In [ ]:
"""
###########################################################
# 5. 모델 정의 (Faster R-CNN + ResNet50 FPN 전이학습)
############################################################

# 사전학습된 Faster R-CNN 모델 로드
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

# 분류 head를 우리 데이터셋 클래스 개수에 맞게 교체
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

model.to(DEVICE)

# 옵티마이저 설정
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(params, lr=1e-4, weight_decay=1e-4)

# (선택) 러닝레이트 스케줄러
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)


############################################################
# 6. 학습 루프
############################################################

num_epochs = 5  # 혹은 원하는 에폭 수
optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

for epoch in range(num_epochs):
    ##########################
    # 1) Train
    ##########################
    model.train()
    train_loss_sum = 0.0

    for images, targets in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        train_loss_sum += losses.item()

    avg_train_loss = train_loss_sum / max(1, len(train_loader))

    ##########################
    # 2) Validation (loss만 측정)
    ##########################
    model.train()  # ★ 여기 중요: eval()이 아니라 train() 상태에서 호출해야 loss 나옴
    val_loss_sum = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss_sum += losses.item()

    avg_val_loss = val_loss_sum / max(1, len(val_loader))

    print(f"[Epoch {epoch+1}/{num_epochs}] "
          f"train_loss: {avg_train_loss:.4f} | val_loss: {avg_val_loss:.4f}")

    # 스케줄러 한 스텝
    scheduler.step()

# 학습된 모델 저장 (원하면)
torch.save(model.state_dict(), "fasterrcnn_oral_drug.pth")
print("모델 저장 완료")

"""

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 131MB/s] 


[Epoch 1/5] train_loss: 0.6592 | val_loss: 0.2819
[Epoch 2/5] train_loss: 0.3678 | val_loss: 0.2162
[Epoch 3/5] train_loss: 0.3111 | val_loss: 0.2247
[Epoch 4/5] train_loss: 0.2328 | val_loss: 0.1078
[Epoch 5/5] train_loss: 0.2152 | val_loss: 0.0869
모델 저장 완료


In [11]:
model_path = os.path.join(extract_path, "exp02-mosaic-SGD.pth") 

In [14]:
# 정답 JSON 로드
with open(TEST_JSON_PATH, "r") as f:
    test_anno_data = json.load(f)

# JSON 안의 파일명 예시 (긴 이름)
json_files = [img["file_name"] for img in test_anno_data["images"]]
# 내 폴더의 파일명 예시 (짧은 이름: 1005.png 등)
my_files = sorted([f for f in os.listdir(TEST_IMG_DIR) if f.endswith(".png")])

print(f"내 파일 샘플: {my_files[:3]}")
print(f"JSON 파일 샘플: {json_files[:3]}")

# 🛠️ 필승 매칭 로직 테스트
fname_to_id = {}
for img in test_anno_data["images"]:
    full_name = img["file_name"]
    # 긴 이름에서 숫자만 추출 (예: '...1005.png' -> '1005')
    pure_num = "".join(filter(str.isdigit, os.path.splitext(full_name)[0]))
    fname_to_id[pure_num] = img["id"]

# 내 파일 하나(예: '1005.png')가 매칭되는지 확인
test_f = my_files[0]
pure_f = "".join(filter(str.isdigit, os.path.splitext(test_f)[0]))
if pure_f in fname_to_id:
    print(f"✅ 매칭 성공! 내 파일 '{test_f}' -> JSON ID '{fname_to_id[pure_f]}'")
else:
    print(f"❌ 매칭 실패! '{pure_f}'라는 숫자를 JSON에서 찾을 수 없습니다.")


내 파일 샘플: ['1.png', '10.png', '100.png']
JSON 파일 샘플: ['K-001900-016551-024850-027926_0_2_0_2_75_000_200.png', 'K-001900-016551-024850-027926_0_2_0_2_90_000_200.png', 'K-001900-016551-024850-029345_0_2_0_2_90_000_200.png']
❌ 매칭 실패! '1'라는 숫자를 JSON에서 찾을 수 없습니다.


In [35]:
############################################################
# 7. test_images 예측 (1280 복구 + ID 매핑)
############################################################

import cv2
import numpy as np
import torch
from tqdm import tqdm

model.eval()
rows = []
# ◀ 문턱값 0.05 (안전하게 설정)
THR = 0.05 

with torch.no_grad():
    # ◀ [:10] 같은 제한 없이 전체 test_files를 돌립니다.
    for f in tqdm(test_files, desc="843장 최종 질주 중"):
        img_path = os.path.join(TEST_IMG_DIR, f)
        orig_img = cv2.imread(img_path)
        if orig_img is None: continue
        h_orig, w_orig = orig_img.shape[:2]

        # 전처리 및 추론 (고속 로직)
        input_img = cv2.resize(orig_img, (640, 640))
        input_img = cv2.cvtColor(input_img, cv2.COLOR_BGR2RGB)
        img_tensor = input_img.astype(np.float32) / 255.0
        img_tensor = (img_tensor - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
        img_tensor = torch.from_numpy(img_tensor).permute(2, 0, 1).unsqueeze(0).to(DEVICE).float()

        outputs_list = model(img_tensor) # 모델은 리스트를 반환함
        outputs = outputs_list[0]         # ◀ 핵심: 리스트에서 첫 번째 결과(딕셔너리)를 꺼냄

        scores = outputs["scores"].cpu().numpy()
        keep = scores >= THR
        
        boxes = outputs["boxes"].cpu().numpy()[keep]
        labels = outputs["labels"].cpu().numpy()[keep]
        scores = scores[keep]

        # ◀ ID 매칭 로직 (아까 1500.0 성공했던 그 로직)
        my_num = "".join(filter(str.isdigit, os.path.splitext(f)[0]))
        # 만약 파일명이 1.png 라면 JSON 인덱스 0번을 가져오도록 보정
        json_idx = str(int(my_num) - 1) 
        real_coco_id = fname_to_id.get(json_idx)

        # ◀ 중요: 만약 ID를 못 찾으면 화면에 찍어서 범인을 잡습니다.
        if real_coco_id is None:
            # print(f"⚠️ ID 매칭 실패: {f}") # 너무 많으면 주석 해제해서 확인
            continue

        ratio = w_orig / 640
        for box, lab, sc in zip(boxes, labels, scores):
            x1, y1, x2, y2 = box * ratio
            
            rows.append({
                "image_id": real_coco_id,
                "category_id": int(model2orig[int(lab)]),
                "bbox_x": float(x1), "bbox_y": float(y1),
                "bbox_w": float(x2 - x1), "bbox_h": float(y2 - y1),
                "score": float(sc)
            })

# 결과 정리 및 저장
df_sub = pd.DataFrame(rows)
if not df_sub.empty:
    df_sub = df_sub.sort_values(by=["image_id", "score"], ascending=[True, False])
    df_sub = df_sub.groupby("image_id").head(4)
    df_sub.insert(0, "annotation_id", range(1, len(df_sub) + 1))
    df_sub.to_csv(os.path.join(extract_path, f"sub_{config['run_name']}.csv"), index=False)
    print(f"\n🎉 성공! 총 {len(df_sub)}개의 객체를 찾았습니다.")
else:
    print("\n❌ 여전히 데이터가 없습니다. fname_to_id 매핑을 다시 확인해야 합니다.")

843장 최종 질주 중: 100%|██████████| 843/843 [02:00<00:00,  6.98it/s]


🎉 성공! 총 4개의 객체를 찾았습니다.


In [32]:
# 방금 찾은 4개의 데이터가 진짜 ID를 가졌는지 확인
print(df_sub[['image_id', 'category_id', 'score']])

   image_id  category_id     score
0    1500.0        27925  0.986081
1    1500.0        16550  0.983796
2    1500.0        24849  0.975160
3    1500.0         1899  0.970597


In [ ]:
"""
############################################################
# 7. test_images에 대해 예측 → submission.csv 생성
############################################################

# test 이미지 파일 목록 가져오기
test_files = [f for f in os.listdir(TEST_IMG_DIR) if f.endswith(".png")]
test_files = sorted(test_files)

model.eval()

rows = []
annotation_id = 1      # submission용 annotation_id 시작
score_threshold = 0.3  # 너무 낮은 점수는 제거 (필요에 따라 조정)

with torch.no_grad():
    for f in test_files:
        img_path = os.path.join(TEST_IMG_DIR, f)
        image = Image.open(img_path).convert("RGB")

        # image_id = 파일명에서 확장자 제거한 문자열 그대로 사용
        image_id = os.path.splitext(f)[0]

        img_tensor = T.ToTensor()(image).to(DEVICE)
        outputs = model([img_tensor])[0]

        keep = outputs["scores"].cpu() >= score_threshold
        boxes  = outputs["boxes"].cpu()[keep]
        labels = outputs["labels"].cpu()[keep]
        scores = outputs["scores"].cpu()[keep]

        for box, lab, sc in zip(boxes, labels, scores):
            x1, y1, x2, y2 = box.tolist()
            w = x2 - x1
            h = y2 - y1

            orig_cat = model2orig[int(lab.item())]

            rows.append({
                "annotation_id": annotation_id,
                "image_id": image_id,  # 문자열 그대로 사용
                "category_id": orig_cat+1,
                "bbox_x": x1,
                "bbox_y": y1,
                "bbox_w": w,
                "bbox_h": h,
                "score": float(sc.item()),
            })

            annotation_id += 1

# DataFrame으로 만들고 저장
df_sub = pd.DataFrame(rows, columns=[
    "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"
])
# 이미지 ID별로 점수 높은 순 정렬 후 상위 4개 추출
df_sub = df_sub.sort_values(by=["image_id", "score"], ascending=[True, False])
df_sub = df_sub.groupby("image_id").head(4)

# 3) 최종 annotation_id 부여 (1부터 순차적으로)
df_sub.insert(0, "annotation_id", range(1, len(df_sub) + 1))

# 4) CSV 저장
output_path = "base_submission.csv"
df_sub.to_csv(os.path.join(extract_path, output_path), index=False)

print(f"✅ 생성 완료: {output_path}")
print(f"📊 총 예측 객체 수: {len(df_sub)}")
print(df_sub.head())


✅ 생성 완료: base_submission.csv
📊 총 예측 객체 수: 3252
   annotation_id image_id  category_id      bbox_x      bbox_y      bbox_w  \
0              1        1        27926  600.338806  669.911072  260.493408   
1              2        1         1900  155.593613  249.810211  205.826401   
2              3        1        16551  557.501282   79.214638  397.296753   
3              4        1        24850  168.999191  729.885376  185.028427   
4              5       10         1900  638.812378  844.820801  194.058960   

       bbox_h     score  
0  486.611633  0.990676  
1  129.355652  0.989452  
2  397.456810  0.985325  
3  315.124390  0.964617  
4  192.124512  0.987158  


In [27]:
if wandb.run is None:
    wandb.init(project="Pill-터링Project_초급프로젝트", name=config["run_name"])

In [28]:
############################################################
# 8. mAP 측정 및 실험 마무리
############################################################

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json

# 1. COCO 평가용 리스트 변환
eval_results = []
for _, row in df_sub.iterrows():
    eval_results.append({
        "image_id": int(row["image_id"]),
        "category_id": int(row["category_id"]),
        "bbox": [row["bbox_x"], row["bbox_y"], row["bbox_w"], row["bbox_h"]],
        "score": float(row["score"])
    })

# 2. 임시 JSON 저장
with open("temp_eval.json", "w") as f:
    json.dump(eval_results, f)

# 3. 평가 실행
print(f"📊 [{config['run_name']}] mAP 성능 평가 시작...")
coco_gt = COCO(TEST_JSON_PATH)
coco_dt = coco_gt.loadRes("temp_eval.json")
coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.evaluate(); coco_eval.accumulate(); coco_eval.summarize()

# 4. WandB 최종 기록 및 종료
wandb.log({
    "final_mAP_50_95": coco_eval.stats[0],
    "final_mAP_50": coco_eval.stats[1],
    "final_mAR_100": coco_eval.stats[8]
})
print(f"\n🎯 최종 mAP @ IoU 0.50: {coco_eval.stats[1]:.4f}")
wandb.finish()



📊 [exp02-mosaic-SGD] mAP 성능 평가 시작...
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.88s).
Accumulating evaluation results...
DONE (t=0.29s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.001
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.001
 Average Recall     (AR) @

final_mAP_50,▁
final_mAP_50_95,▁
final_mAR_100,▁
final_mAP_50,0.00014
final_mAP_50_95,4e-05
final_mAR_100,0.00092


In [ ]:
"""
############################################################
# 8. 모델 성능 평가 (mAP 측정)
############################################################

import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# 1. df_sub(Pandas)를 COCO 평가용 리스트로 변환
eval_results = []
for _, row in df_sub.iterrows():
    eval_results.append({
        "image_id": int(row["image_id"]),
        "category_id": int(row["category_id"])-1,
        "bbox": [row["bbox_x"], row["bbox_y"], row["bbox_w"], row["bbox_h"]],
        "score": float(row["score"])
    })

# 2. 임시 JSON 파일로 저장
temp_json = "temp_eval.json"
with open(temp_json, "w") as f:
    json.dump(eval_results, f)

# 3. COCO 평가 실행
coco_gt = COCO(TEST_JSON_PATH)
coco_dt = coco_gt.loadRes(temp_json)

coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()
"""

loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.43s).
Accumulating evaluation results...
DONE (t=0.48s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.392
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.419
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.417
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.392
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.936
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.936
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe


결론부터 말씀드리면, 현재 모델은 **"정답은 기가 막히게 잘 찾아내지만(Recall), 위치를 정밀하게 그리는 능력(Precision)은 아직 보완이 필요하다"** 고 요약할 수 있습니다.

<br>

### **📊 핵심 지표 분석 (Performance Review)**

#### **1. Precision vs Recall의 불균형**

* **mAP (IoU 0.50:0.95):** **0.374**
* **mAR (maxDets 100):** **0.898**

* **해석:** 리콜(Recall)이 **0.898**로 매우 높습니다. 이는 모델이 이미지 속 물체를 **거의 놓치지 않고 다 찾아내고 있다**는 뜻입니다. 반면 AP는 그에 비해 낮습니다. 즉, 물체를 찾긴 찾는데 **Bounding Box의 위치가 약간 어긋나 있거나, 오탐(False Positive)이 섞여 있을 확률**이 높습니다.

<br>

#### **2. IoU 임계값(Threshold)에 따른 변화**

* **AP @ 0.50:** **0.397**
* **AP @ 0.75:** **0.395**

    * **해석:** 보통 IoU 기준이 엄격해질수록(0.5 $\rightarrow$ 0.75) 점수가 팍 떨어지기 마련인데, 이 모델은 거의 차이가 없습니다.
    * **진단:** 이는 모델이 **Box를 아주 대충 그리거나, 아주 정확하게 그리거나 둘 중 하나**라는 뜻입니다. 어중간하게 틀리는 게 아니라 위치 정확도 면에서 어떤 일관된 특징(예: 항상 실제보다 조금 크게 그림)이 있을 수 있습니다.

<br>

#### **3. Object Size에 따른 성능 (중요!)**

* **Small / Medium:** **-1.000**
* **Large:** **0.374**

    * **해석:** `-1.000`은 해당 데이터가 **평가셋에 아예 존재하지 않는다**는 뜻입니다.
    * **진단:** 현재 재찬 님이 다루시는 데이터셋의 물체들은 모두 **'Large'** 카테고리에 속합니다. 작은 물체를 탐지할 걱정은 안 해도 되지만, 큰 물체의 전체적인 윤곽을 더 정밀하게 잡는 데 집중해야 합니다.

<br>

### **💡 1타 강사의 '성능 향상' 클리닉**

재찬 님, 점수를 더 올리고 싶다면 수강생들에게 다음 **3가지 튜닝 전략**을 가르쳐주며 적용해 보세요.

1. **Localization Loss 강화:** 현재 리콜은 충분하니, 박스 위치를 더 정확히 잡아야 합니다. `CIoU`나 `DIoU` 같은 Loss 함수를 쓰고 있는지 확인하고, Box Regression의 가중치를 조금 높여보세요.

2. **Confidence Threshold 튜닝:** 리콜이 0.9에 가깝다는 건 모델이 아주 자신 있게(혹은 너무 남발해서) 박스를 치고 있다는 겁니다. Confidence 임계값을 살짝 높여서 확실한 것만 남기면 AP(정밀도)가 올라갈 수 있습니다.

3. **데이터 증강(Augmentation) 점검:** 물체가 크기 때문에(Large), 이미지 가장자리에 걸린 물체를 잘 잡는지 확인해 보세요. `Random Crop` 보다는 `Scaling`이나 `Translation` 위주의 증강이 더 효과적일 수 있습니다.

<br>

### **🎯 최종 리포트**

> **"모델이 눈은 좋은데(Recall), 손재주가 살짝 부족합니다(Precision). 박스를 더 예쁘게 그리도록 가르치면 점수는 수직 상승할 겁니다!"**